# Maia Fine-tuning (Gemma 4 E4B → Maia 2026)

**Automated pipeline using Colab MCP + HF MCP**

- Base: `unsloth/gemma-4-E4B-it`
- Dataset: 133K+ examples + 2025-2026 knowledge updates
- Method: QLoRA 4-bit + LoRA r=16
- Hardware: Free T4 GPU (~10-12h)
- Output: 16-bit HF + GGUF (Ollama)

In [ ]:
import os
from getpass import getpass

# Configuration
BASE_MODEL = 'unsloth/gemma-4-E4B-it'
MAX_SEQ_LEN = 8192
INFERENCE_CTX = 131072  # 128K
EPOCHS = 1
LR = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4

# HF Token (from Secrets)
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    HF_TOKEN = getpass('HuggingFace token (read+write): ')
    os.environ['HF_TOKEN'] = HF_TOKEN

In [ ]:
# Install dependencies
!pip install -q -U unsloth
!pip install -q -U datasets trl peft accelerate bitsandbytes huggingface_hub
!pip install -q sentencepiece protobuf

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=True)
print('✓ HF login OK')

In [ ]:
# Clone Maia repo & prepare dataset
!git clone --depth 1 https://github.com/calitosaa/Maia /content/maia_repo
%cd /content/maia_repo/finetuning/output
!cat maia_gemma4_finetune.jsonl.part_* > maia_base.jsonl
!wc -l maia_base.jsonl maia_knowledge_2025_2026.jsonl

In [ ]:
import json
from datasets import Dataset

# Load base + updates
examples = []
for fname in ['maia_base.jsonl', 'maia_knowledge_2025_2026.jsonl']:
    with open(fname) as f:
        for line in f:
            try:
                ex = json.loads(line)
                if 'messages' in ex and len(ex['messages']) >= 2:
                    examples.append({'messages': ex['messages']})
            except:
                continue

ds = Dataset.from_list(examples)
print(f'Total dataset: {len(ds)} examples')
print(f'Sample: {ds[0]}')

In [ ]:
# Load Gemma 4 E4B with QLoRA 4-bit
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

# Set 128K context for inference
model.config.max_position_embeddings = INFERENCE_CTX
tokenizer.model_max_length = INFERENCE_CTX
print(f'✓ Model loaded: train ctx={MAX_SEQ_LEN}, inference ctx={INFERENCE_CTX}')

In [ ]:
# Apply LoRA (r=16)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# Apply Gemma chat template
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='gemma')

def format_examples(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

ds = ds.map(format_examples, batched=True, remove_columns=['messages'])
print(f'✓ Dataset formatted\nSample:\n{ds[0]["text"][:300]}')

In [ ]:
# Mount Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/maia_checkpoints_2026'
!mkdir -p $OUTPUT_DIR

In [ ]:
# Train with SFTTrainer
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=20,
    save_steps=500,
    save_total_limit=3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    optim='adamw_8bit',
    weight_decay=0.01,
    seed=42,
    report_to='none',
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=config,
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Training complete!\n{trainer_stats}')

In [ ]:
# Get username for repo naming
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
USERNAME = api.whoami()['name']
REPO_ID = f'{USERNAME}/maia-gemma4-e4b-2026'
print(f'Username: {USERNAME}')
print(f'Repo ID: {REPO_ID}')

In [ ]:
# Save merged 16-bit + push to HF
model.save_pretrained_merged(
    f'{OUTPUT_DIR}/maia-final',
    tokenizer,
    save_method='merged_16bit'
)
model.push_to_hub_merged(
    REPO_ID,
    tokenizer,
    save_method='merged_16bit',
    token=HF_TOKEN
)
print(f'✓ 16-bit model pushed: https://huggingface.co/{REPO_ID}')

In [ ]:
# Convert to GGUF Q4_K_M
model.save_pretrained_gguf(
    f'{OUTPUT_DIR}/maia-gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
model.push_to_hub_gguf(
    f'{REPO_ID}-GGUF',
    tokenizer,
    quantization_method='q4_k_m',
    token=HF_TOKEN
)
print(f'✓ GGUF model pushed: https://huggingface.co/{REPO_ID}-GGUF')

In [ ]:
# Test inference
FastLanguageModel.for_inference(model)
messages = [
    {'role': 'user', 'content': '¿Quién eres y qué puedes hacer? Responde brevemente.'}
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors='pt'
).to('cuda')
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9
)
result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print('\n=== Test Inference ===')
print(result)
print('\n✓ Training complete! Models uploaded to HuggingFace Hub.')